# 4. Differential expression and enrichment

Which genes differ between **ANCA-GN** and
the other glomerulonephritis types, within a cell type? We use **pseudobulk** so the
**sample** is the unit of replication, then ask which pathways the changes touch.

### Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import stats
import course_utils as cu
sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. Define the groups

In [ ]:
adata = cu.load_annotated()
adata.obs['group'] = np.where(adata.obs['disease'] == 'ANCA-GN', 'ANCA-GN', 'other-GN')
print(adata.obs.groupby(['group','disease','sample'], observed=True).size())

## 2. Pseudobulk

> **Method note: why pseudobulk.** Cells from one biopsy are not
independent, so testing per cell (tens of thousands of them) hugely overstates significance
(pseudoreplication). Summing counts to one profile per sample x cell type and testing across
**samples** respects the real replication. With 5 ANCA-GN vs 3 other-GN samples this is
underpowered, so read it as exploratory, and remember there is no healthy Control on this
slide.

In [ ]:
# pseudobulk: sum raw counts per (sample, cell type). The SAMPLE is the replicate.
C = adata.layers['counts'].tocsr()
groups = adata.obs.groupby(['sample', 'celltype'], observed=True).indices
rows, meta = [], []
for (samp, ct), idx in groups.items():
    if len(idx) < 25:            # skip tiny groups
        continue
    rows.append(np.asarray(C[idx].sum(0)).ravel()); meta.append((samp, ct, len(idx)))
pb = pd.DataFrame(np.vstack(rows), columns=adata.var_names)
pbm = pd.DataFrame(meta, columns=['sample', 'celltype', 'n_cells'])
samp2grp = adata.obs.drop_duplicates('sample').set_index('sample')['group']
pbm['group'] = pbm['sample'].map(samp2grp).values
# CPM + log for the pseudobulk profiles
pbn = np.log1p(pb.div(pb.sum(1), axis=0) * 1e4)
print('pseudobulk profiles:', pbn.shape[0], '(sample x celltype)')

## 3. DE within a cell type

> **Method note.** We contrast the proximal tubule (the
largest epithelial compartment, where injury is prominent in GN). A Welch t-test on the
log-CPM pseudobulk per gene, with Benjamini-Hochberg FDR. A dedicated tool (DESeq2/edgeR via
pseudobulk, or limma-voom) is preferable in production; the t-test keeps us dependency-light
and transparent.

In [ ]:
# DE within one cell type: ANCA-GN vs other GN, sample-level t-test per gene
CT = 'PT'
sel = pbm['celltype'] == CT
a = pbn[sel & (pbm['group'] == 'ANCA-GN').values]
b = pbn[sel & (pbm['group'] == 'other-GN').values]
print(f'{CT}: {len(a)} ANCA-GN vs {len(b)} other-GN sample-pseudobulks')
t, p = stats.ttest_ind(a.values, b.values, equal_var=False)
lfc = a.mean(0) - b.mean(0)
res = pd.DataFrame({'gene': adata.var_names, 'log2FC_like': lfc.values, 't': t, 'p': p})
res['fdr'] = stats.false_discovery_control(np.nan_to_num(res['p'], nan=1.0))
res = res.sort_values('p')
res.head(12)

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(res['log2FC_like'], -np.log10(res['p'].clip(lower=1e-300)), s=10, alpha=0.5, color='0.6')
sig = res['fdr'] < 0.1
plt.scatter(res.loc[sig, 'log2FC_like'], -np.log10(res.loc[sig, 'p']), s=14, color='#c0392b')
for _, r in res.head(8).iterrows():
    plt.annotate(r['gene'], (r['log2FC_like'], -np.log10(max(r['p'], 1e-300))), fontsize=7)
plt.axvline(0, color='0.7', lw=1); plt.xlabel('mean log-CPM diff (ANCA-GN - other)')
plt.ylabel('-log10 p'); plt.title(f'{CT}: ANCA-GN vs other GN (pseudobulk)'); plt.tight_layout(); plt.show()

> **Method note: transcript spillover (segmentation).** Watch for genes that are not
native to a cell type, e.g. immune/chemokine genes such as CX3CR1 or CCR2 turning up in the
proximal-tubule pseudobulk above. In dense tissue the default Xenium segmentation (a nuclear
mask grown by a fixed expansion) assigns transcripts from neighbouring cells to the wrong
cell, so a cell type's profile is contaminated by its neighbours. This *spillover* inflates
cross-lineage signal and can produce spurious differential expression. Mitigations: re-segment
with an image-based method such as **Cellpose** (Stringer et al., Nat Methods 2021), or with
transcript-aware methods that reassign molecules by their spatial pattern, **Baysor**
(Petukhov et al., Nat Biotechnol 2022) and **Proseg** (Jones et al. 2024), then recompute. As
a rule, treat cross-lineage genes in a cell type's DE with suspicion.

## 4. Enrichment of the up-regulated genes

> **Method note.** With a 480-gene panel, a
full pathway enrichment (GSEA on MSigDB) is not meaningful, the gene universe is tiny and
curated. So we do a transparent **over-representation test** (hypergeometric) of the
up-genes against a few curated, panel-restricted programmes. In production with whole-
transcriptome data use `decoupler`/`gseapy` with MSigDB. Read these as 'which programme is
over-represented', not as exact p-values.

In [ ]:
# over-representation of the up-genes in curated, panel-restricted gene sets (hypergeometric)
sets = {
 'Inflammation/cytokine': ['IL1B','TNF','IL6','CXCL9','CXCL10','CXCL11','CCL2','CCL5','CXCL8','IFNG','STAT1','CXCL2','CXCL3'],
 'Complement': ['C3','C1QB','C1QC','CFH','C5AR1','C7','C5','C4A'],
 'ECM/fibrosis': ['COL1A1','COL3A1','COL6A3','FN1','VCAN','LUM','TIMP1','MMP9','MMP14','ACTA2','TGFB1'],
 'Endothelial activation': ['VCAM1','ICAM1','SELE','SELP','PLVAP','ANGPT2'],
 'Tubule injury': ['HAVCR1','VCAM1','CLU','SPP1','LCN2'],
}
bg = set(adata.var_names)
sets = {k: [g for g in v if g in bg] for k, v in sets.items()}
up = set(res[(res['fdr'] < 0.1) & (res['log2FC_like'] > 0)]['gene'])
if len(up) < 5:
    up = set(res[res['log2FC_like'] > 0].head(40)['gene'])   # fall back to top up-genes
N = len(bg); n = len(up)
rows = []
for name, gs in sets.items():
    K = len(gs); k = len(up & set(gs))
    pval = stats.hypergeom.sf(k - 1, N, K, n) if k > 0 else 1.0
    rows.append({'set': name, 'in_set': K, 'hits': k, 'genes': ','.join(sorted(up & set(gs))), 'p': round(pval, 4)})
pd.DataFrame(rows).sort_values('p')

> **Exercise.** Repeat the pseudobulk DE for the immune compartment (`IMM`). Which genes
separate ANCA-GN from the other GN types there?

In [ ]:
# Solution: repeat the pseudobulk DE for the immune compartment
CT = 'IMM'
sel = pbm['celltype'] == CT
a = pbn[sel & (pbm['group'] == 'ANCA-GN').values]; b = pbn[sel & (pbm['group'] == 'other-GN').values]
t, p = stats.ttest_ind(a.values, b.values, equal_var=False)
out = pd.DataFrame({'gene': adata.var_names, 'diff': (a.mean(0)-b.mean(0)).values, 'p': p}).sort_values('p')
print(out.head(10).to_string(index=False))

> **Exercise (ML).** As an ML aside, treat the pseudobulk profiles as a tiny classification
problem: fit an **L1-regularised logistic regression** to separate ANCA-GN from other-GN within
one cell type, and score it with **leave-one-out** cross-validation. With only 8 samples this
overfits badly, the exercise is to *see* how unstable the accuracy and the selected genes are,
and why per-sample n, not per-cell n, is what limits you.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_score
sel = pbm['celltype'] == 'PT'
X = pbn[sel.values].values
y = (pbm.loc[sel, 'group'] == 'ANCA-GN').astype(int).values
m = LogisticRegression(penalty='l1', solver='liblinear', C=0.5)
acc = cross_val_score(m, X, y, cv=LeaveOneOut()).mean()
m.fit(X, y)
nz = np.asarray(adata.var_names)[m.coef_[0] != 0]
print(f'PT, {len(y)} samples | leave-one-out accuracy = {acc:.2f} (interpret with great caution)')
print(f'{len(nz)} genes selected (changes a lot if you drop a sample):', list(nz)[:15])

### Recap

Pseudobulk DE (sample as replicate) avoids pseudoreplication; a curated ORA
reads the programmes behind the changes. Small n and no Control mean these are hypotheses.
Next: the spatial layer, what is next to what.